In [ ]:
import os, re
from pathlib import Path
from collections import defaultdict
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

try:
    from IPython.display import display
except ImportError:
    def display(df):
        print(df.to_string(index=False))

pd.set_option('display.max_rows', None)
pd.set_option('display.width', 160)

ROOT            = Path('data')
CHBMIT_DIR      = ROOT / 'chb-mit'
SIENA_DIR       = ROOT / 'siena'
SEIZEIT_DIR     = ROOT / 'seizeit2'
MENDELEY_DIR    = ROOT / 'mendeley'
SEIZEIT_SESSION = 'ses-01'
MIN_CONTEXTS    = 2

# ── Grade de parametros ───────────────────────────────────────────────────────
# (GUARD_min, MIN_INTER_min, MAX_PRE_min)
# GUARD  = zona proibida pos-crise (e usada como margem antes tambem)
# MIN_INTER = interictal minimo necessario
# MAX_PRE   = janela pre-ictal maxima testada
#             -> onset deve ser >= MAX_PRE + GUARD para contexto valido
#             (precisamos de MAX_PRE de pre-ictal limpo antes do onset)
PARAM_GRID = [
    (30, 15, 30),
    (45, 30, 45),
    (45, 30, 30),
    (60, 30, 45),
    (60, 45, 45),
    (90, 45, 45),
]
DEFAULT_PARAMS = (45, 30, 45)

# ── Ground truth Siena ────────────────────────────────────────────────────────
SIENA_SEIZURE_GT = {
    'PN00': {
        'pn00-1.edf':[(1143,1213)],'pn00-2.edf':[(1220,1274)],
        'pn00-3.edf':[(765,4425)],'pn00-4.edf':[(1006,1080)],'pn00-5.edf':[(904,971)],
    },
    'PN01': {
        'pn01.edf':  [(10218,10272),(46353,46427)],
        'pn01-1.edf':[(10218,10272),(46353,46427)],
    },
    'PN03': {'pn03-1.edf':[(38673,38784)],'pn03-2.edf':[(34921,35054)]},
    'PN05': {'pn05-2.edf':[(7163,7198)],'pn05-3.edf':[(6836,6866)],'pn05-4.edf':[(3608,3647)]},
    'PN06': {
        'pno6-1.edf':[(5583,5647)],'pno6-2.edf':[(8860,8929)],
        'pn06-3.edf':[(6275,6317)],'pno6-4.edf':[(5939,6002)],'pn06-5.edf':[(4783,4827)],
        'pn06-1.edf':[(5583,5647)],'pn06-2.edf':[(8860,8929)],'pn06-4.edf':[(5939,6002)],
    },
    'PN07': {'pn07-1.edf':[(22059,22121)]},
    'PN09': {'pn09-1.edf':[(7249,7329)],'pn09-2.edf':[(7127,7186)],'pn09-3.edf':[(7221,7285)]},
    'PN10': {
        'pn10-1.edf':[(7545,7614)],'pn10-2.edf':[(7798,7849)],'pn10-3.edf':[(7835,7904)],
        'pn10-4.5.6.edf':[(2309,2314),(6544,6563),(11225,11282)],
        'pn10-7.8.9.edf':[(2748,2796),(5459,5477),(12923,12938)],
        'pn10-10.edf':[(7977,7991)],
    },
    'PN11': {'pn11-.edf':[(7554,7609)],'pn11-1.edf':[(7554,7609)]},
    'PN12': {
        'pn12-1.2.edf':[(1312,1375),(9570,9638)],
        'pn12-3.edf':[(772,868)],'pn12-4.edf':[(9812,9875)],
    },
    'PN13': {'pn13-1.edf':[(7062,7110)],'pn13-2.edf':[(7249,7314)],'pn13-3.edf':[(7553,7704)]},
    'PN14': {
        'pn14-1.edf':[(7262,7289)],'pn14-2.edf':[(7479,7491)],
        'pn14-3.edf':[(17540,17581)],'pn14-4.edf':[(5463,5546)],
    },
    'PN16': {'pn16-1.edf':[(7184,7307)],'pn16-2.edf':[(8574,8681)]},
    'PN17': {'pn17-1.edf':[(8420,8490)],'pn17-2.edf':[(7731,7814)]},
}

MENDELEY_SEIZURE_GT = {
    'p10': {'p10_record1.edf':[(7199,7644)],'p10_record2.edf':[(7200,7505)]},
    'p11': {
        'p11_record1.edf':[(4565,4629)],'p11_record2.edf':[(6372,6423)],
        'p11_record3.edf':[(1517,1608),(4397,4480),(7201,7277),(9263,9336)],
        'p11_record4.edf':[(7177,8535)],
    },
    'p12': {
        'p12_record1.edf':[(5756,5832),(7194,7298)],
        'p12_record2.edf':[(7202,7320),(10777,10859)],
        'p12_record3.edf':[(196,278),(2474,2638),(5426,5539)],
    },
    'p13': {
        'p13_record1.edf':[(7196,7248)],'p13_record2.edf':[(3241,3266),(7196,7212)],
        'p13_record3.edf':[(7240,7258)],'p13_record4.edf':[(1450,1480),(6651,6675)],
    },
    'p14': {
        'p14_record1.edf':[(3466,3494),(7216,7350)],
        'p14_record2.edf':[(1783,1815),(7181,7191)],
        'p14_record3.edf':[(3121,3152),(5519,5564),(5929,5969),(51379,51419)],
    },
    'p15': {
        'p15_record1.edf':[(7158,7208)],'p15_record2.edf':[(7147,7194)],
        'p15_record3.edf':[(7132,7145)],'p15_record4.edf':[(2431,2487),(7234,7254)],
    },
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def normalize_fn(fname):
    return re.sub(r'-+', '-', fname.lower())

def siena_match(fn, iv_map):
    fn_low = normalize_fn(fn)
    if fn_low in iv_map: return iv_map[fn_low]
    parts = fn_low.split('-', 1)
    if len(parts) == 2:
        p, s = parts
        for v in [p.replace('0','o'), p.replace('o','0')]:
            key = v + '-' + s
            if key in iv_map: return iv_map[key]
    return []

def parse_chbmit_summary(path):
    sz_map, free = {}, []
    with open(path, encoding='utf-8', errors='ignore') as f:
        blocks = f.read().split('File Name:')
    for block in blocks[1:]:
        lines = block.strip().split('\n')
        fname = lines[0].strip()
        n_sz = 0; onsets = []; offsets = []
        for ln in lines:
            if 'Number of Seizures' in ln:
                try: n_sz = int(re.search(r'\d+', ln.split(':',1)[1]).group())
                except: pass
            elif 'Seizure' in ln and 'Start Time' in ln:
                m = re.search(r'(\d+)', ln.split(':',1)[1])
                if m: onsets.append(int(m.group()))
            elif 'Seizure' in ln and 'End Time' in ln:
                m = re.search(r'(\d+)', ln.split(':',1)[1])
                if m: offsets.append(int(m.group()))
        if n_sz > 0:
            sz_map[fname] = [(o,e) for o,e in zip(onsets,offsets) if e>o]
        else:
            free.append(fname)
    return sz_map, free

# ── Validacao de contextos ────────────────────────────────────────────────────
def validate_contexts(edf_list, dataset, guard_s, min_inter_s, max_pre_s, shared_s=600):
    """
    Criterios para contexto valido:
    1. onset >= max_pre_s + guard_s
       (precisa de max_pre_s de pre-ictal + guard_s de margem antes disso)
    2. Interictal disponivel >= min_inter_s
       (regiao antes do guard_antes, ou depois do guard_pos, ou dedicado)
    """
    DEDICATED = dataset in ('CHBMIT', 'SeizeIT2')
    pure_total = sum(e['dur_s'] for e in edf_list if e['is_pure'])
    total_sz   = sum(len(e['seizures']) for e in edf_list)

    contexts = []
    for edf in edf_list:
        if edf['is_pure'] or not edf['seizures']:
            continue
        dur  = edf['dur_s']
        seiz = edf['seizures']
        n_sz = len(seiz)

        for idx, (on, off) in enumerate(seiz):
            # Para datasets com interictal DEDICADO (CHB-MIT, SeizeIT2), o interictal
            # nao vem do mesmo arquivo/regiao, entao o unico requisito de espaco e
            # caber o pre-ictal (on >= max_pre_s). Para Siena/Mendeley (interictal
            # intra-arquivo, antes da propria crise) precisamos tambem do guard
            # entre o interictal e o pre-ictal: on >= max_pre_s + guard_s.
            pre_req = max_pre_s if DEDICATED else (max_pre_s + guard_s)
            pre_disponivel = on
            cabe_pre = pre_disponivel >= pre_req
            max_pre_real = max(0, min(on, max_pre_s)) if DEDICATED else max(0, on - guard_s)

            if idx == 0:
                r1 = on - pre_req
            else:
                prev_off = seiz[idx-1][1]
                inicio_zona = prev_off + guard_s
                r1 = (on - pre_req) - inicio_zona

            if idx == n_sz - 1:
                r3 = dur - (off + guard_s)
            else:
                next_on = seiz[idx+1][0]
                fim_zona = next_on - pre_req
                r3 = fim_zona - (off + guard_s)

            r1 = max(r1, 0); r3 = max(r3, 0)

            per_ctx = 0
            if pure_total > 0 and total_sz > 0:
                per_ctx = max(0, (pure_total - shared_s*(total_sz-1)) / total_sz)

            if per_ctx >= min_inter_s:
                source, avail = 'dedicado', per_ctx
                has_inter = True
            elif r1 >= min_inter_s:
                source, avail = 'antes', r1
                has_inter = True
            elif r3 >= min_inter_s:
                source, avail = 'depois', r3
                has_inter = True
            elif (r1 + r3) >= min_inter_s:
                source, avail = 'R1+R3', r1+r3
                has_inter = True
            else:
                source, avail = 'insuf', r1+r3+per_ctx
                has_inter = False

            valid = cabe_pre and has_inter

            if not cabe_pre and not has_inter:
                reason = f'onset={on/60:.0f}min < necessario={pre_req/60:.0f}min | inter insuf={avail/60:.0f}min'
            elif not cabe_pre:
                reason = f'onset={on/60:.0f}min < necessario={pre_req/60:.0f}min'
            elif not has_inter:
                reason = f'inter insuf: R1={r1/60:.0f} R3={r3/60:.0f} ded={per_ctx/60:.0f}min'
            else:
                reason = f'{source} inter={avail/60:.0f}min pre_real={max_pre_real/60:.0f}min'

            contexts.append({
                'edf': edf['name'], 'sz_idx': idx+1,
                'on_s': on, 'off_s': off, 'dur_sz_s': off-on,
                'max_pre_real_s': max_pre_real,
                'cabe_pre': cabe_pre,
                'source': source, 'avail_s': avail,
                'has_inter': has_inter,
                'valid': valid, 'reason': reason,
            })
    return contexts

def n_valid_contexts(contexts):
    return sum(1 for c in contexts if c['valid'])

# ── Leitores ──────────────────────────────────────────────────────────────────
def _dur_s(p, fs=256, ch=23):
    try: return p.stat().st_size / (fs * ch * 2)
    except: return 3600.0

def load_chbmit():
    patients = {}
    if not CHBMIT_DIR.exists(): return patients
    for pat_dir in sorted(d for d in CHBMIT_DIR.iterdir() if d.is_dir()):
        pat = pat_dir.name
        summary = pat_dir / f'{pat}-summary.txt'
        if not summary.exists(): continue
        sz_map, free_list = parse_chbmit_summary(summary)
        disk = {f.name: f for f in pat_dir.glob('*.edf')}
        edf_list = []
        for fname, iv in sz_map.items():
            if fname not in disk: continue
            edf_list.append({'name':fname,'dur_s':_dur_s(disk[fname]),'seizures':iv,'is_pure':False})
        for fname in free_list:
            if fname not in disk: continue
            edf_list.append({'name':fname,'dur_s':_dur_s(disk[fname]),'seizures':[],'is_pure':True})
        patients[pat] = edf_list
    return patients

def load_siena():
    patients = {}
    if not SIENA_DIR.exists(): return patients
    for pat_dir in sorted(d for d in SIENA_DIR.iterdir() if d.is_dir()):
        pat = pat_dir.name
        if pat not in SIENA_SEIZURE_GT: continue
        gt = SIENA_SEIZURE_GT[pat]
        edf_list = []
        for edf_path in sorted(pat_dir.glob('*.edf')):
            fn  = edf_path.name
            iv  = siena_match(fn, gt)
            try:
                import mne
                raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
                dur_s = raw.n_times / raw.info['sfreq']; raw.close()
            except:
                dur_s = _dur_s(edf_path, fs=512, ch=29)
            edf_list.append({'name':fn,'dur_s':dur_s,'seizures':iv,'is_pure':len(iv)==0})
        patients[pat] = edf_list
    return patients

def load_seizeit2():
    patients = {}
    if not SEIZEIT_DIR.exists(): return patients
    for pat_dir in sorted(d for d in SEIZEIT_DIR.iterdir()
                          if d.is_dir() and d.name.startswith('sub-')):
        pat     = pat_dir.name
        eeg_dir = pat_dir / SEIZEIT_SESSION / 'eeg'
        if not eeg_dir.exists(): continue
        edf_list = []
        for edf_path in sorted(eeg_dir.glob('*_eeg.edf')):
            fn  = edf_path.name
            tsv = eeg_dir / fn.replace('_eeg.edf','_events.tsv')
            iv  = []
            if tsv.exists():
                try:
                    df = pd.read_csv(tsv, sep='\t')
                    if 'eventType' in df.columns:
                        et = df['eventType'].astype(str).str.lower().str.strip()
                        for _, row in df[et.str.startswith('sz')].iterrows():
                            try:
                                o = float(row['onset']); d = float(row['duration'])
                                if d > 0: iv.append((o, o+d))
                            except: pass
                except: pass
            try:
                import mne
                raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
                dur_s = raw.n_times / raw.info['sfreq']; raw.close()
            except:
                dur_s = _dur_s(edf_path, fs=256, ch=2)
            edf_list.append({'name':fn,'dur_s':dur_s,'seizures':iv,'is_pure':len(iv)==0})
        patients[pat] = edf_list
    return patients

def load_mendeley():
    patients = {}
    edf_dir = MENDELEY_DIR / 'Raw_EDF_Files'
    if not edf_dir.exists(): edf_dir = MENDELEY_DIR
    if not edf_dir.exists(): return patients
    by_pat = defaultdict(list)
    for p in sorted(edf_dir.glob('*.edf')):
        m = re.match(r'(p\d+)', p.name, re.IGNORECASE)
        if m: by_pat[m.group(1).lower()].append(p)
    for pat, paths in sorted(by_pat.items()):
        gt = MENDELEY_SEIZURE_GT.get(pat, {})
        edf_list = []
        for edf_path in sorted(paths):
            fn  = edf_path.name
            iv  = gt.get(fn.lower(), [])
            try:
                import mne
                raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
                dur_s = raw.n_times / raw.info['sfreq']; raw.close()
            except:
                dur_s = _dur_s(edf_path, fs=500, ch=21)
            edf_list.append({'name':fn,'dur_s':dur_s,'seizures':iv,'is_pure':len(iv)==0})
        patients[pat] = edf_list
    return patients

# ── Excel helpers ─────────────────────────────────────────────────────────────
HDR  = PatternFill('solid', fgColor='2C2C2A')
HFNT = Font(color='F1EFE8', bold=True, size=10)
YES  = PatternFill('solid', fgColor='EAF3DE')
NO   = PatternFill('solid', fgColor='FCEBEB')
ALT  = PatternFill('solid', fgColor='F1EFE8')
PUR  = PatternFill('solid', fgColor='E6F1FB')
AMB  = PatternFill('solid', fgColor='FAEEDA')
GRNN = PatternFill('solid', fgColor='1D9E75')
thin = Side(style='thin', color='D3D1C7')
BRD  = Border(left=thin, right=thin, top=thin, bottom=thin)
CTR  = Alignment(horizontal='center', vertical='center', wrap_text=True)

def hc(ws, r, c, v, w=None, fill=HDR, font=HFNT):
    cell = ws.cell(r, c, v)
    cell.fill = fill; cell.font = font; cell.alignment = CTR; cell.border = BRD
    if w: ws.column_dimensions[get_column_letter(c)].width = w

def dc(ws, r, c, v, fill=None):
    cell = ws.cell(r, c, v)
    cell.alignment = CTR; cell.border = BRD
    if fill: cell.fill = fill

# ── Aba Resumo ────────────────────────────────────────────────────────────────
def write_summary(wb, all_patients):
    ws = wb.create_sheet('Resumo', 0)
    ws.freeze_panes = 'B3'
    DATASETS = ['CHBMIT','Siena','SeizeIT2','Mendeley']
    DS_LBL   = {'CHBMIT':'CHB-MIT','Siena':'Siena','SeizeIT2':'SeizeIT2','Mendeley':'Mendeley'}

    for ci, (lbl, w) in enumerate([('Dataset',14),('Pac disco',12),
                                    ('EDFs crise',12),('EDFs sem crise',14),('Total crises',12)], 1):
        hc(ws, 1, ci, lbl, w)

    p0 = 6
    for ci, (g, mi, mp) in enumerate(PARAM_GRID):
        col = p0 + ci
        is_default = (g, mi, mp) == DEFAULT_PARAMS
        lbl = f'G={g} I={mi} Pre={mp}\nPac sel (>={MIN_CONTEXTS}ctx)'
        cell = ws.cell(1, col, lbl)
        cell.fill = GRNN if is_default else HDR
        cell.font = HFNT; cell.alignment = CTR; cell.border = BRD
        ws.column_dimensions[get_column_letter(col)].width = 16
    ws.row_dimensions[1].height = 52

    totals = {'edfs_sz':0,'edfs_free':0,'crises':0,'sel':[0]*len(PARAM_GRID)}

    for ri, ds in enumerate(DATASETS, 2):
        pats       = all_patients.get(ds, {})
        edfs_sz    = sum(1 for el in pats.values() for e in el if not e['is_pure'] and e['seizures'])
        edfs_free  = sum(1 for el in pats.values() for e in el if e['is_pure'])
        n_crises   = sum(len(e['seizures']) for el in pats.values() for e in el)
        fill_row   = ALT if ri % 2 == 0 else PatternFill()

        dc(ws, ri, 1, DS_LBL[ds], fill_row)
        dc(ws, ri, 2, len(pats), fill_row)
        dc(ws, ri, 3, edfs_sz, fill_row)
        dc(ws, ri, 4, edfs_free, PUR)
        dc(ws, ri, 5, n_crises, fill_row)

        totals['edfs_sz'] += edfs_sz
        totals['edfs_free'] += edfs_free
        totals['crises'] += n_crises

        for ci, (g, mi, mp) in enumerate(PARAM_GRID):
            col = p0 + ci
            guard_s = g*60; min_i_s = mi*60; max_pre_s = mp*60
            n_sel = 0
            for pat, edf_list in pats.items():
                ctxs = validate_contexts(edf_list, ds, guard_s, min_i_s, max_pre_s)
                if n_valid_contexts(ctxs) >= MIN_CONTEXTS:
                    n_sel += 1
            totals['sel'][ci] += n_sel
            fill = YES if n_sel >= 8 else (AMB if n_sel >= 4 else NO)
            dc(ws, ri, col, f'{n_sel}/{len(pats)}', fill)

    ri_tot = len(DATASETS) + 2
    total_pats = sum(len(p) for p in all_patients.values())
    for ci, v in enumerate([('TOTAL',None),(total_pats,None),(totals['edfs_sz'],None),
                             (totals['edfs_free'],None),(totals['crises'],None)], 1):
        hc(ws, ri_tot, ci, v[0])
    for ci, v in enumerate(totals['sel']):
        hc(ws, ri_tot, p0+ci, v)

    rl = ri_tot + 2
    ws.cell(rl,   1, 'Legenda:').font = Font(bold=True, size=10)
    ws.cell(rl+1, 1, f'Coluna verde = parametro padrao G={DEFAULT_PARAMS[0]} I={DEFAULT_PARAMS[1]} Pre={DEFAULT_PARAMS[2]}')
    ws.cell(rl+2, 1, 'Verde escuro (celula) = >=8 pac sel | Amarelo = 4-7 | Vermelho = <4')
    ws.cell(rl+3, 1, 'G=GUARD_min (zona proib pos+margem antes), I=MIN_INTER_min, Pre=MAX_PRE_min')
    ws.cell(rl+4, 1, 'Criterio valido: onset >= Pre+G  E  interictal >= I')
    ws.column_dimensions['A'].width = 16

# ── Aba detalhada ─────────────────────────────────────────────────────────────
COLS = [
    ('dataset',          'Dataset',           10),
    ('paciente',         'Paciente',          10),
    ('edf',              'Arquivo EDF',       46),
    ('dur_edf_min',      'Dur EDF (min)',      12),
    ('tem_crise',        'Tem crise?',         10),
    ('n_crises',         'N crises',            8),
    ('crise_idx',        'Crise #',             8),
    ('onset_min',        'Onset (min)',         12),
    ('offset_min',       'Offset (min)',        12),
    ('dur_crise_min',    'Dur crise (min)',     14),
    ('max_pre_real_min', 'Pre disp (min)',      13),
    ('cabe_pre',         'Cabe pre-ictal?',     14),
    ('inter_fonte',      'Fonte inter',         20),
    ('inter_disp_min',   'Inter disp (min)',    14),
    ('ctx_valido',       'Ctx valido?',         13),
    ('ctx_val_pac',      'Ctx val pac',         11),
    ('pac_sel',          'Pac sel?',            10),
    ('obs',              'Observacao',          55),
]

def write_detail(wb, name, rows):
    ws = wb.create_sheet(name)
    ws.freeze_panes = 'A2'
    for ci, (_, lbl, w) in enumerate(COLS, 1):
        hc(ws, 1, ci, lbl, w)
    ws.row_dimensions[1].height = 32

    prev_pat = None; alt = False
    for ri, row in enumerate(rows, 2):
        pat = row.get('paciente','')
        if pat != prev_pat: alt = not alt; prev_pat = pat
        base = ALT if alt else PatternFill()
        for ci, (key, _, _) in enumerate(COLS, 1):
            val = row.get(key, '')
            cell = ws.cell(ri, ci, val)
            cell.alignment = CTR; cell.border = BRD
            if key == 'ctx_valido':
                cell.fill = YES if val=='Sim' else (NO if val=='Nao' else PUR)
            elif key == 'cabe_pre':
                cell.fill = YES if val=='Sim' else (NO if val=='Nao' else PUR)
            elif key == 'pac_sel':
                cell.fill = YES if val=='Sim' else (NO if val=='Nao' else PatternFill())
            elif key == 'tem_crise' and val=='Nao':
                cell.fill = PUR
            else:
                cell.fill = base

def build_rows(ds_lbl, pat, edf_list, contexts, n_valid):
    rows = []
    ctx_map = defaultdict(list)
    for c in contexts: ctx_map[c['edf']].append(c)
    sel = 'Sim' if n_valid >= MIN_CONTEXTS else 'Nao'

    for edf in edf_list:
        fn   = edf['name']
        seiz = edf['seizures']
        dur  = round(edf['dur_s']/60, 2)
        ctxs = ctx_map.get(fn, [])

        if seiz:
            for i, (on, off) in enumerate(seiz):
                ctx = next((c for c in ctxs if c['sz_idx']==i+1), None)
                rows.append({
                    'dataset': ds_lbl, 'paciente': pat, 'edf': fn,
                    'dur_edf_min': dur,
                    'tem_crise': 'Sim', 'n_crises': len(seiz), 'crise_idx': i+1,
                    'onset_min':  round(on/60, 2),
                    'offset_min': round(off/60, 2),
                    'dur_crise_min': round((off-on)/60, 2),
                    'max_pre_real_min': round(ctx['max_pre_real_s']/60, 1) if ctx else '--',
                    'cabe_pre':   'Sim' if (ctx and ctx['cabe_pre']) else ('Nao' if ctx else '--'),
                    'inter_fonte':    ctx['source'] if ctx else '--',
                    'inter_disp_min': round(ctx['avail_s']/60, 1) if ctx else '--',
                    'ctx_valido': 'Sim' if (ctx and ctx['valid']) else ('Nao' if ctx else '--'),
                    'ctx_val_pac': n_valid, 'pac_sel': sel,
                    'obs': ctx['reason'] if ctx else '',
                })
        else:
            rows.append({
                'dataset': ds_lbl, 'paciente': pat, 'edf': fn,
                'dur_edf_min': dur,
                'tem_crise':'Nao','n_crises':0,'crise_idx':'--',
                'onset_min':'--','offset_min':'--','dur_crise_min':'--',
                'max_pre_real_min':'--','cabe_pre':'--',
                'inter_fonte': 'dedicado' if ds_lbl in ('CHB-MIT','SeizeIT2') else '--',
                'inter_disp_min': dur,
                'ctx_valido':'--','ctx_val_pac':n_valid,'pac_sel':sel,
                'obs':'EDF sem crise',
            })
    return rows

# ── Selecao automatica: apenas pacientes que passam no criterio LOSO (>=2 ctx) ─
DS_LBL = {'CHBMIT': 'CHB-MIT', 'Siena': 'Siena', 'SeizeIT2': 'SeizeIT2', 'Mendeley': 'Mendeley'}

# Quantidade final de pacientes por dataset (sempre os de maior n de contextos
# validos, dentre os que ja passam o criterio n_valid >= MIN_CONTEXTS).
TOP_N = {
    'CHBMIT':   7,
    'Siena':    7,
    'SeizeIT2': 7,
    'Mendeley': 4,
}

# Em vez de uma lista fixa, cada paciente do disco e avaliado com os parametros
# padrao; so entra na selecao final quem tiver n_contextos_validos >= MIN_CONTEXTS.
# Dentro de cada dataset, os selecionados sao ordenados do maior para o menor
# numero de contextos (para facilitar a leitura, nao limita quantidade).

def compute_all_contexts(all_patients, guard_s, min_i_s, max_pre_s):
    """Retorna {dataset: {paciente: n_valid}} para todos os pacientes do disco."""
    result = {}
    for ds, pats in all_patients.items():
        result[ds] = {}
        for pat, edf_list in pats.items():
            ctxs = validate_contexts(edf_list, ds, guard_s, min_i_s, max_pre_s)
            result[ds][pat] = n_valid_contexts(ctxs)
    return result

def build_selected_summary(all_patients, guard_s, min_i_s, max_pre_s):
    """
    Seleciona automaticamente, por dataset, os TOP_N pacientes com maior
    n_contextos_validos, dentre os que passam o criterio n_valid >= MIN_CONTEXTS
    (obrigatorio para LOSO). Retorna tambem a contagem de candidatos elegiveis
    antes do corte, para se saber se algum dataset tinha poucos aprovados.
    """
    all_n_valid = compute_all_contexts(all_patients, guard_s, min_i_s, max_pre_s)

    rows = []
    n_elegiveis = {}
    for ds, pats in all_patients.items():
        candidatos = []
        for pat, edf_list in pats.items():
            n_valid = all_n_valid[ds][pat]
            if n_valid < MIN_CONTEXTS:
                continue  # nao passa no criterio LOSO -- fora da selecao
            ctxs = validate_contexts(edf_list, ds, guard_s, min_i_s, max_pre_s)
            edfs_com_ctx = sorted({c['edf'] for c in ctxs if c['valid']})
            candidatos.append({
                'Dataset': DS_LBL[ds],
                'Paciente': pat,
                'N EDFs total': len(edf_list),
                'N crises total': sum(len(e['seizures']) for e in edf_list),
                'N contextos validos': n_valid,
                'EDFs com contexto valido': ', '.join(edfs_com_ctx) if edfs_com_ctx else '--',
                'Serve p/ LOSO (>=2)?': 'Sim',
            })
        candidatos.sort(key=lambda r: r['N contextos validos'], reverse=True)
        n_elegiveis[DS_LBL[ds]] = len(candidatos)
        n_top = TOP_N.get(ds, len(candidatos))
        candidatos = candidatos[:n_top]
        rows.extend(candidatos)

    df = pd.DataFrame(rows)
    df.attrs['n_elegiveis'] = n_elegiveis  # candidatos totais antes do corte TOP_N
    return df


def print_selected_analysis(all_patients):
    g, mi, mp = DEFAULT_PARAMS
    guard_s, min_i_s, max_pre_s = g * 60, mi * 60, mp * 60

    df = build_selected_summary(all_patients, guard_s, min_i_s, max_pre_s)
    n_elegiveis = df.attrs.get('n_elegiveis', {})

    print(f'\n=== Top-N selecionados (maior n de contextos, dentre quem passa LOSO >=2) '
          f'— parametros G={g} I={mi} Pre={mp} ===\n')
    print(f'Corte TOP_N configurado: {TOP_N}\n')
    if df.empty:
        print('Nenhum paciente passou no criterio com esses parametros.')
        return df
    display(df)

    print('\n=== Contagem de pacientes selecionados por dataset ===\n')
    resumo = df.groupby('Dataset').agg(
        n_pacientes_selecionados=('Paciente', 'count'),
        min_ctx=('N contextos validos', 'min'),
        max_ctx=('N contextos validos', 'max'),
        media_ctx=('N contextos validos', lambda s: round(s.mean(), 2)),
    ).reset_index()
    resumo['N candidatos elegiveis (antes do corte)'] = resumo['Dataset'].map(n_elegiveis)
    display(resumo)

    # Alerta se algum dataset tinha menos elegiveis do que o TOP_N pedido
    for ds_key, n_top in TOP_N.items():
        lbl = DS_LBL[ds_key]
        n_disp = n_elegiveis.get(lbl, 0)
        if n_disp < n_top:
            print(f'ATENCAO: {lbl} pedia top-{n_top} mas so tinha {n_disp} '
                  f'candidato(s) elegivel(is) (>=2 ctx). Selecionado(s) todos os {n_disp}.')

    total = len(df)
    print(f'\nTotal geral de pacientes selecionados: {total}')

    return df

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print('[Carregando...]', flush=True)
    loaders = [
        ('CHBMIT',   load_chbmit,   'CHB-MIT'),
        ('Siena',    load_siena,    'Siena'),
        ('SeizeIT2', load_seizeit2, 'SeizeIT2'),
        ('Mendeley', load_mendeley, 'Mendeley'),
    ]
    all_patients = {}
    for ds, loader, lbl in loaders:
        print(f'  {lbl}...', flush=True)
        all_patients[ds] = loader()
        print(f'    {len(all_patients[ds])} pacientes', flush=True)

    wb = Workbook(); wb.remove(wb.active)

    print('[Resumo...]', flush=True)
    write_summary(wb, all_patients)

    g, mi, mp = DEFAULT_PARAMS
    guard_s = g*60; min_i_s = mi*60; max_pre_s = mp*60

    for ds, _, lbl in loaders:
        print(f'[Aba {lbl}...]', flush=True)
        pats = all_patients[ds]
        all_rows = []
        for pat in sorted(pats):
            edf_list = pats[pat]
            ctxs     = validate_contexts(edf_list, ds, guard_s, min_i_s, max_pre_s)
            n_valid  = n_valid_contexts(ctxs)
            rows     = build_rows(lbl, pat, edf_list, ctxs, n_valid)
            all_rows.extend(rows)
            print(f'    {pat}: {n_valid} ctx validos', flush=True)
        write_detail(wb, lbl, all_rows)

    out = ROOT / 'auditoria_datasets.xlsx'
    wb.save(str(out))
    print(f'\nSalvo em: {out}', flush=True)

    # Analise dos pacientes selecionados (usa o mesmo all_patients ja carregado)
    print_selected_analysis(all_patients)

if __name__ == '__main__':
    main()

[Carregando...]
  CHB-MIT...


    23 pacientes
  Siena...


C:\Users\danil\AppData\Local\Temp\ipykernel_1916\2765878710.py:278: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
C:\Users\danil\AppData\Local\Temp\ipykernel_1916\2765878710.py:278: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
C:\Users\danil\AppData\Local\Temp\ipykernel_1916\2765878710.py:278: RuntimeWarning: Highpass cutoff frequency 15.91549 is greater than lowpass cutoff frequency 15.0, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
C:\Users\danil\AppData\Local\Temp\ipykernel_1916\2765878710.py:278: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
C:\User

    14 pacientes
  SeizeIT2...


C:\Users\danil\AppData\Local\Temp\ipykernel_1916\2765878710.py:278: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
C:\Users\danil\AppData\Local\Temp\ipykernel_1916\2765878710.py:278: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
C:\Users\danil\AppData\Local\Temp\ipykernel_1916\2765878710.py:278: RuntimeWarning: Highpass cutoff frequency 15.91549 is greater than lowpass cutoff frequency 7.0, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
C:\Users\danil\AppData\Local\Temp\ipykernel_1916\2765878710.py:278: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose=False)
C:\Users